In [4]:
!pip install --upgrade pip
!pip install --upgrade skfolio
!pip install yfinance

In [13]:
# ============================================
# HERC + Pre-Selection + Walk-Forward (Yahoo)
# con RandomizedSearchCV & MultipleRandomizedCV
# ============================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import datetime as dt
import yfinance as yf

from plotly.io import show
from sklearn import set_config
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
import scipy.stats as stats

from skfolio import Population, RatioMeasure, RiskMeasure
from skfolio.metrics import make_scorer
from skfolio.model_selection import MultipleRandomizedCV, WalkForward, cross_val_predict
from skfolio.moments import ShrunkCovariance
from skfolio.optimization import HierarchicalEqualRiskContribution
from skfolio.pre_selection import SelectKExtremes
from skfolio.preprocessing import prices_to_returns
from skfolio.prior import EmpiricalPrior

# -----------------------------
# CONFIG
# -----------------------------
set_config(transform_output="pandas")

# 👉 Sostituisci con il tuo universo. Se lasci pochi tickers, il codice si adatta.
TICKERS = [
    "AAPL","MSFT","NVDA","GOOGL","AMZN",
    "META","AVGO","TSLA","LLY","JPM","XOM","UNH","V","MA","HD","PG","KO","PEP","COST",
    "ADBE","NFLX","CRM","INTC","CSCO","AMD","LIN","ABBV","BAC","TMO","WMT","PFE","NKE",
    "MCD","ORCL","TXN","AMAT","IBM","QCOM","PM","HON","UPS","CAT","GE","MS","GS","BLK",
    "NOW","BKNG","DE","SPGI","AXP","CVX","COP","MDLZ","SBUX","INTU","ISRG","ELV","ADP",
    "CI","LMT","BA","RTX"
]  # ~64 (simile all’esempio FTSE)

START = "2000-01-04"  # coerente con l'esempio
END   = "2025-09-26"  # oggi (adatta se vuoi un backtest chiuso)

# -----------------------------
# DATA (Yahoo Finance)
# -----------------------------
def download_adj_close(tickers, start, end):
    df = yf.download(tickers, start=start, end=end, auto_adjust=False, progress=False)["Adj Close"]
    if isinstance(df, pd.Series):
        df = df.to_frame()
    df = df.dropna(how="all").ffill().dropna(how="any")
    # uniforma colonne
    df.columns = [str(c).upper() for c in df.columns]
    return df

prices = download_adj_close(TICKERS, START, END)
returns = prices_to_returns(prices)       # log-returns per skfolio

# Sequential split 67/33 (no shuffle)
X_train, X_test = train_test_split(returns, test_size=0.33, shuffle=False)
print(f"Train: {X_train.index.min().date()} → {X_train.index.max().date()} | "
      f"Test: {X_test.index.min().date()} → {X_test.index.max().date()}")
print(f"Universe size: {X_train.shape[1]} assets")

# -----------------------------
# PIPELINE: Pre-Selection + HERC
# -----------------------------
# Pre-selezione: top-k per Sharpe
# (k e shrinkage li tuneremo con RandomizedSearchCV)
pre_selection = SelectKExtremes(k=min(10, X_train.shape[1]), measure=RatioMeasure.SHARPE_RATIO, highest=True)

# HERC con covarianza shrunk
optimization = HierarchicalEqualRiskContribution(
    prior_estimator=EmpiricalPrior(covariance_estimator=ShrunkCovariance(shrinkage=0.5)),
    risk_measure=RiskMeasure.VARIANCE,
)

model_bench = Pipeline([
    ("pre_selection", pre_selection),
    ("optimization", optimization),
])

# -----------------------------
# Rebalancing: Walk-Forward
# -----------------------------
# Mensile: 20 giorni di test, 252 di train (1 anno)
walk_forward = WalkForward(test_size=20, train_size=252)

# -----------------------------
# Hyper-Parameter Tuning (RandomizedSearchCV)
# -----------------------------
# Adattiamo gli spazi di ricerca alla dimensione del tuo universo
n_assets = X_train.shape[1]
k_low, k_high = max(5, n_assets//12), max(10, n_assets//2)
if k_low >= k_high:
    k_low, k_high = max(3, n_assets//3), max(5, (2*n_assets)//3)

param_distributions = {
    "pre_selection__k": stats.randint(low=k_low, high=min(k_high, n_assets)),
    "optimization__prior_estimator__covariance_estimator__shrinkage": stats.uniform(0, 1),
}

random_search = RandomizedSearchCV(
    estimator=model_bench,
    cv=walk_forward,
    n_jobs=-1,
    param_distributions=param_distributions,
    n_iter=30,                   # aumenta per ricerche più robuste
    random_state=0,
    scoring=make_scorer(RatioMeasure.CVAR_RATIO),  # massimizza CVaR Ratio out-of-sample
)

random_search.fit(X_train)
model_tuned = random_search.best_estimator_
print("Best params:", random_search.best_params_)

# -----------------------------
# Standard Walk-Forward Analysis (una singola traiettoria)
# -----------------------------
pred_bench = cross_val_predict(model_bench, X_test, cv=walk_forward)
pred_bench.name = "Benchmark Model"

pred_tuned = cross_val_predict(model_tuned, X_test, cv=walk_forward, n_jobs=-1)
pred_tuned.name = "Tuned Model"

population = Population([pred_bench, pred_tuned])

# Cumulative returns (Plotly fig)
fig_cum = population.plot_cumulative_returns()
show(fig_cum)

# Summary delle metriche principali
print("\n=== Standard Walk-Forward Summary ===")
print(population.summary())

# -----------------------------
# Multiple Randomized Cross-Validation (Resampling)
# -----------------------------
# Se l’universo è piccolo, riduciamo la dimensione del sottocampione e la finestra
if n_assets >= 40:
    asset_subset = min(50, n_assets)           # fino a 50 asset
else:
    asset_subset = max( min(20, n_assets), 8)  # tra 8 e 20 in universi piccoli

window_size = 3 * 252 if len(X_test) >= 3*252 else max(252, len(X_test)//2)

cv_mc = MultipleRandomizedCV(
    walk_forward=walk_forward,
    n_subsamples=200,              # 500 nell'esempio; aumenta se vuoi più robustezza
    asset_subset_size=asset_subset,
    window_size=window_size,
    random_state=0,
)

pred_bench_mc = cross_val_predict(
    model_bench,
    X_test,
    cv=cv_mc,
    n_jobs=-1,
    portfolio_params={"tag": "Benchmark Model"},
)

pred_tuned_mc = cross_val_predict(
    model_tuned,
    X_test,
    cv=cv_mc,
    n_jobs=-1,
    portfolio_params={"tag": "Tuned Model"},
)

population_mc = pred_bench_mc + pred_tuned_mc

# Prime 10 traiettorie cumulative del modello tuned
fig_first10 = pred_tuned_mc[:10].plot_cumulative_returns(use_tag_in_legend=False)
show(fig_first10)

# Distribuzione degli Sharpe ann. out-of-sample
fig_dist = population_mc.plot_distribution(
    measure_list=[RatioMeasure.ANNUALIZED_SHARPE_RATIO],
    tag_list=["Benchmark Model", "Tuned Model"],
)
show(fig_dist)

# Media e deviazione standard degli Sharpe
for pred in [pred_bench_mc, pred_tuned_mc]:
    tag = pred[0].tag
    mean_sr = pred.measures_mean(measure=RatioMeasure.ANNUALIZED_SHARPE_RATIO)
    std_sr = pred.measures_std(measure=RatioMeasure.ANNUALIZED_SHARPE_RATIO)
    print(f"\n{tag}\n{'=' * len(tag)}")
    print(f"Average Sharpe Ratio: {mean_sr:0.2f}")
    print(f"Sharpe Ratio Std Dev: {std_sr:0.2f}")

# Box plot del CVaR Ratio
fig_box = population_mc.boxplot_measure(
    measure=RatioMeasure.CVAR_RATIO, tag_list=["Benchmark Model", "Tuned Model"]
)
show(fig_box)

# Composizione per i primi 2 MultiPeriodPortfolio del modello tuned
fig_comp = pred_tuned_mc[:2].plot_composition(display_sub_ptf_name=False)
show(fig_comp)

# Evoluzione dei pesi per la prima traiettoria
fig_w = pred_tuned_mc[0].plot_weights_per_observation()
show(fig_w)


Train: 2013-01-03 → 2021-07-12 | Test: 2021-07-13 → 2025-09-25
Universe size: 63 assets
Best params: {'optimization__prior_estimator__covariance_estimator__shrinkage': np.float64(0.4499499899112276), 'pre_selection__k': 29}



=== Standard Walk-Forward Summary ===
                                   Benchmark Model Tuned Model
Mean                                        0.082%      0.093%
Annualized Mean                             20.71%      23.32%
Variance                                    0.018%      0.013%
Annualized Variance                          4.49%       3.26%
Semi-Variance                              0.0094%     0.0066%
Annualized Semi-Variance                     2.37%       1.67%
Standard Deviation                           1.33%       1.14%
Annualized Standard Deviation               21.18%      18.05%
Semi-Deviation                               0.97%       0.81%
Annualized Semi-Deviation                   15.40%      12.92%
Mean Absolute Deviation                      0.93%       0.82%
CVaR at 95%                                  3.10%       2.53%
EVaR at 95%                                  4.93%       3.96%
Worst Realization                            7.26%       6.61%
CDaR at 95%     


Benchmark Model
Average Sharpe Ratio: 1.42
Sharpe Ratio Std Dev: 0.39

Tuned Model
Average Sharpe Ratio: 1.34
Sharpe Ratio Std Dev: 0.36
